# The HotpotQA environment, by hand

One pass through everything the evolution loop does *before* it edits a graph: build an environment, look at a task,
call its tools yourself, run the solver loop, and rank the resulting trajectories the way `evolve` ranks them.
No evolution machinery is used.

- **Part 1 (offline, no API key):** construct the environment, call tools by hand, drive the real solver loop with a scripted fake model.
- **Part 2 (live, about one cent in total):** a small batch with the real solver, ranked; the same batch under the expert graph; and the refiner prompt that *would* be sent.

Needs `uv run pg gen-data hotpotqa` once. Run from the repo with e.g. `uv run --with jupyterlab jupyter lab analysis/`.

## 1. Construct the environment

`make_env` is the only place a concrete scenario is chosen. Everything after this line talks to the `Environment` / `Episode` interface in `pg/envs/base.py`.

In [1]:
from pg.config import ROOT, Config
from pg.envs import make_env

cfg = Config.from_env()            # reads .env (models, concurrency); no network yet
env = make_env("hotpotqa", cfg)

print(type(env).__name__, "| max_steps =", env.max_steps)
print({split: len(env.tasks(split)) for split in ("train", "val", "test")})
print("\nsystem prompt the solver gets:\n" + env.system_prompt())

HotpotQAEnv | max_steps = 8
{'train': 100, 'val': 50, 'test': 150}

system prompt the solver gets:
You answer multi-hop questions using tools over a fixed set of context paragraphs.
Use `search` and `lookup` to gather evidence, then call `finish` with a short answer span
(an entity name, a date, a number, or yes/no), not a sentence.


## 2. A task

`prompt` is all the solver sees. `meta` is the hidden side: the gold answer and the question's own ten paragraphs (two relevant, eight distractors).

In [2]:
task = env.tasks("train")[0]
print(task.id)
print(task.prompt)
print("gold answer:", task.meta["answer"], "| type:", task.meta["type"], "| level:", task.meta["level"])
for p in task.meta["paragraphs"]:
    print(f"  [{p['title']}] {p['text'][:70]}...")

5a8fab8c5542995b4424208a
Question: Who set Nietzche's philosophical novel to music? 
gold answer: Richard Strauss | type: bridge | level: hard
  [Thus Spoke Zarathustra] Thus Spoke Zarathustra: A Book for All and None (German: "Also sprach ...
  [Ishmael (novel)] Ishmael is a 1992 philosophical novel by Daniel Quinn. It examines the...
  [The Fall (Camus novel)] The Fall (French: La Chute ) is a philosophical novel by Albert Camus....
  [Sidney (novel)] Sidney is a philosophical novel by the American writer Margaret Deland...
  [Theologus Autodidactus] Theologus Autodidactus ("The Self-taught Theologian"), originally titl...
  [The Time of the Angels] The Time of the Angels is a philosophical novel by British novelist Ir...
  [Also sprach Zarathustra (Strauss)] Also sprach Zarathustra , Op. 30 (Thus Spoke Zarathustra or Thus Spake...
  [Marius the Epicurean] Marius the Epicurean: his sensations and ideas is a historical and phi...
  [Hermsprong] Hermsprong: or, Man As He Is Not is a 17

## 3. An episode, driven by hand

`env.start(task)` returns a fresh `Episode`. Its tools are closures over that episode, which is why parallel rollouts cannot interfere. Here we play the agent ourselves.

In [3]:
episode = env.start(task)
tools = {t.name: t for t in episode.tools}
for t in episode.tools:
    print(f"{t.name}{tuple(t.args)}: {t.description}")

search('query',): Search the context paragraphs. Returns the 3 paragraphs with the highest word overlap with the query.
lookup('title',): Return the full paragraph with the given title.
finish('answer',): Submit the final answer (a short span such as an entity name, a date, or yes/no). Ends the episode.


In [4]:
query = " ".join(task.prompt.replace("Question:", "").split()[:6])     # a deliberately lazy query: the first few words
print("search(%r)\n" % query)
print(tools["search"].invoke({"query": query})[:600])

search("Who set Nietzche's philosophical novel to")

[The Fall (Camus novel)] The Fall (French: La Chute ) is a philosophical novel by Albert Camus. First published in 1956, it is his last complete work of fiction. Set in Amsterdam, "The Fall" consists of a series of dramatic monologues by the self-proclaimed "judge-penitent" Jean-Baptiste Clamence, as he reflects upon his life to a stranger. In what amounts to a confession, Clamence tells of his success as a wealthy Parisian defense lawyer who was highly respected by his colleagues; his crisis, and his ultimate "fall" from grace, was meant to invoke, in secular terms, The Fall of Man in the Gar


In [5]:
title = task.meta["paragraphs"][0]["title"]
print(tools["lookup"].invoke({"title": title})[:300])
print("\ndone yet?", episode.is_done())

[Thus Spoke Zarathustra] Thus Spoke Zarathustra: A Book for All and None (German: "Also sprach Zarathustra: Ein Buch für Alle und Keinen" , also translated as Thus Spake Zarathustra) is a philosophical novel by German philosopher Friedrich Nietzsche, composed in four parts between 1883 and 1885 and 

done yet? False


In [6]:
print(tools["finish"].invoke({"answer": task.meta["answer"]}))   # cheat: submit the gold answer
print("done?", episode.is_done())
episode.score()

Answer submitted.
done? True


{'score': 1.0,
 'success': True,
 'em': 1.0,
 'f1': 1.0,
 'answer': 'Richard Strauss',
 'gold': 'Richard Strauss'}

`score()` is deterministic: SQuAD-style token F1 against the gold answer, `success` = F1 >= 0.5. A second `env.start(task)` starts from scratch:

In [7]:
fresh = env.start(task)
{t.name: t for t in fresh.tools}["finish"].invoke({"answer": "a wrong answer"})
fresh.score()

{'score': 0.0,
 'success': False,
 'em': 0.0,
 'f1': 0.0,
 'answer': 'a wrong answer',
 'gold': 'Richard Strauss'}

## 4. The solver loop, with a scripted model (still offline)

`run_episode` is the real loop (`guide -> solver -> tools`, a LangGraph `StateGraph`). Instead of an LLM we hand it a
fake chat model that replays a fixed script of tool calls - the same trick the unit tests use. What comes back is a
`Trajectory`: the record everything downstream is built on.

In [8]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from pg.agent import run_episode

class ScriptedModel(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self

def call(name, **args):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": f"call_{name}"}])

script = [call("search", query=query), call("lookup", title=title), call("finish", answer=task.meta["answer"])]
trajectory = run_episode(env, task, None, cfg, solver=ScriptedModel(messages=iter(script)))   # graph=None: no guidance

print("score", trajectory.score, "| success", trajectory.success, "| error", trajectory.error)
for s in trajectory.steps:
    print(f"  step {s.index}: {s.tool}({s.args}) -> {s.observation[:70]!r}")

score 1.0 | success True | error None
  step 0: search({'query': "Who set Nietzche's philosophical novel to"}) -> '[The Fall (Camus novel)] The Fall (French: La Chute ) is a philosophic'
  step 1: lookup({'title': 'Thus Spoke Zarathustra'}) -> '[Thus Spoke Zarathustra] Thus Spoke Zarathustra: A Book for All and No'
  step 2: finish({'answer': 'Richard Strauss'}) -> 'Answer submitted.'


`compact()` is the one-line-per-step form the refiner reads:

In [9]:
print(trajectory.compact())

task=5a8fab8c5542995b4424208a score=1.00 success=True steps=3
  0. search({"query": "Who set Nietzche's philosophical novel to"}) -> [The Fall (Camus novel)] The Fall (French: La Chute ) is a philosophical novel by Albert Camus. First published in 1956, it is his last complete work of fiction. Set in Amsterdam, "The Fall" consists ...
  1. lookup({"title": "Thus Spoke Zarathustra"}) -> [Thus Spoke Zarathustra] Thus Spoke Zarathustra: A Book for All and None (German: "Also sprach Zarathustra: Ein Buch für Alle und Keinen" , also translated as Thus Spake Zarathustra) is a philosophica...
  2. finish({"answer": "Richard Strauss"}) -> Answer submitted.


## 5. A real batch (live)

Six training questions with the real solver and **no graph**. `run_batch` is a thread pool over `run_episode`; this is
exactly the call `evolve` makes at the start of every round.

In [10]:
from pg.agent import run_batch

batch = env.tasks("train")[:6]
trajectories = run_batch(env, batch, None, cfg)

    5ab680d15542995eadef0011: score=1.00 steps=2


    5a808f96554299485f598655: score=0.50 steps=2


    5a8fab8c5542995b4424208a: score=1.00 steps=2


    5ade66e1554299728e26c713: score=1.00 steps=2


    5ae3dc205542992f92d823a1: score=1.00 steps=2


    5a803d695542996402f6a494: score=0.47 steps=5


In [11]:
cost = lambda t: sum(v for k, v in t.usage.items() if k.endswith("cost"))
for t in trajectories:
    gold = next(x.meta["answer"] for x in batch if x.id == t.task_id)
    print(f"{t.task_id}  score={t.score:.2f}  steps={len(t.steps)}  ${cost(t):.5f}  answered={t.metrics.get('answer')!r}  gold={gold!r}")

5a8fab8c5542995b4424208a  score=1.00  steps=2  $0.00025  answered='Richard Strauss'  gold='Richard Strauss'
5ade66e1554299728e26c713  score=1.00  steps=2  $0.00034  answered='1970s and 1980s'  gold='1970s and 1980s'
5ab680d15542995eadef0011  score=1.00  steps=2  $0.00022  answered='Pleiospilos'  gold='Pleiospilos'
5a808f96554299485f598655  score=0.50  steps=2  $0.00025  answered='the First Balkan War'  gold='the Slavic women accompanying their husbands in the First Balkan War.'
5a803d695542996402f6a494  score=0.47  steps=5  $0.00086  answered='Plateau is a two-player abstract strategy game; Colt Express is a railway-themed family board game.'  gold='Colt Express is a railway-themed'
5ae3dc205542992f92d823a1  score=1.00  steps=2  $0.00024  answered='minor basilica'  gold='a minor basilica'


## 6. Rank them, as `evolve` does

No model judges anything. It is a sort on the environment's own score, with errored episodes dropped, and the top and
bottom `k` become the refiner's evidence (`k = Config.refiner_k = 3`, or half the batch if it is small). With a batch
this small and a task this easy, expect ties: the sort is stable, so ties keep batch order.

In [12]:
ranked = sorted((t for t in trajectories if not t.error), key=lambda t: t.score, reverse=True)
n = max(1, min(cfg.refiner_k, len(ranked) // 2))
best, worst = ranked[:n], ranked[-n:]
print("best :", [(t.task_id[-6:], round(t.score, 2)) for t in best])
print("worst:", [(t.task_id[-6:], round(t.score, 2)) for t in worst])
print("\n--- the lowest-scoring trajectory, as the refiner would read it ---\n")
print(worst[-1].compact())

best : [('24208a', 1.0), ('26c713', 1.0), ('ef0011', 1.0)]
worst: [('d823a1', 1.0), ('598655', 0.5), ('f6a494', 0.47)]

--- the lowest-scoring trajectory, as the refiner would read it ---

task=5a803d695542996402f6a494 score=0.47 success=False steps=5
  0. search({"query": "Plateau game Colt Express main difference gameplay rules objective"}) -> [GetAmped2] GetAmped2 (ゲットアンプド2 , GettoAnpudo 2 ) is a free to play massively multiplayer online Fighting-Action, Role-playing video game, developed and published by the Japanese company CyberStep, In...
  1. search({"query": "\"Plateau\" \"Colt Express\" game"}) -> [Colt Express] Colt Express is a railway-themed family board game designed by Christophe Raimbault, Illustrated by Ian Parovel and Jordi Valbuena, published in 2014 by Ludonaute and distributed by Asm...
  2. lookup({"title": "Plateau (game)"}) -> [Plateau (game)] Plateau is a two-player abstract strategy board game invented by Jim Albea.
  3. lookup({"title": "Colt Express"}) -> [C

## 7. The same batch with a graph

Now the online half switches on: before every solver turn the guidance model sees the graph neighbourhood around the
last tool call and writes advice, which is appended to the system prompt for that turn only. The `Trajectory` keeps
each piece of advice in `guidance_log` and accounts for its tokens separately.

In [13]:
from pg.graph import ProceduralGraph

graph = ProceduralGraph.load(ROOT / "graphs" / "hotpotqa_expert.json")
print(graph.summary())
guided = run_batch(env, batch, graph, cfg)

7 nodes, 10 edges


    5ade66e1554299728e26c713: score=1.00 steps=2


    5a808f96554299485f598655: score=0.50 steps=3


    5ab680d15542995eadef0011: score=1.00 steps=3


    5a8fab8c5542995b4424208a: score=1.00 steps=4


    5ae3dc205542992f92d823a1: score=1.00 steps=3


    5a803d695542996402f6a494: score=0.44 steps=5


In [14]:
g = guided[0]
print(batch[0].prompt, "\n")
for i, advice in enumerate(g.guidance_log):
    step = g.steps[i] if i < len(g.steps) else None
    print(f"guidance before step {i}: {advice}")
    if step:
        print(f"   -> {step.tool}({step.args})\n")

Question: Who set Nietzche's philosophical novel to music?  

guidance before step 0: Begin with the **search** action using the distinctive phrase **“Nietzsche philosophical novel”** (also try the corrected spelling “Nietzsche” rather than “Nietzche”). Do not answer from memory; retrieve the paragraph identifying the novel and its musical setting.
   -> search({'query': 'Nietzsche philosophical novel'})

guidance before step 1: The search identified the relevant work as *Thus Spoke Zarathustra*. Proceed to **analyze**: use that title as the bridge entity and search specifically for who “set *Thus Spoke Zarathustra* to music.” Avoid stopping at identifying the novel; retrieve evidence for the composer/artist.
   -> search({'query': 'Thus Spoke Zarathustra set to music composer artist'})

guidance before step 2: The search results support both hops: Nietzsche’s philosophical novel is *Thus Spoke Zarathustra*, and the composer who set it to music was Richard Strauss. Proceed to verificat

In [15]:
from pg.trajectory import summarize

for name, ts in (("no graph", trajectories), ("expert graph", guided)):
    s = summarize(ts)
    tokens = lambda p: s.get(f"{p}_input_tokens", 0) + s.get(f"{p}_output_tokens", 0)
    print(f"{name:13} mean F1 {s['mean_score']:.3f} | success {s['success_rate']:.0%} | steps {s['mean_steps']:.1f} | "
          f"solver tokens {tokens('solver'):,} | guidance tokens {tokens('guidance'):,} | ${s.get('solver_cost', 0) + s.get('guidance_cost', 0):.4f}")

no graph      mean F1 0.828 | success 83% | steps 2.5 | solver tokens 8,106 | guidance tokens 0 | $0.0022
expert graph  mean F1 0.824 | success 83% | steps 3.3 | solver tokens 12,715 | guidance tokens 21,731 | $0.0106


Six questions prove nothing about which is better (see `analysis/paired_comparison.py` and the README for how to measure that). The point here is the mechanics: same tasks, same loop, one extra LLM call per step.

## 8. What the refiner would be shown

This is where the evolution machinery would take over. We stop one step short: build the exact prompt `evolve` would
send, starting from an edgeless skeleton graph, but do not send it.

In [16]:
from pg.refiner import build_prompt

skeleton = ProceduralGraph.skeleton("hotpotqa_demo", env.tool_descriptions())
print(skeleton.summary(), "\n")
prompt = build_prompt(env.description, env.tool_descriptions(), skeleton, best, worst, rejected=[],
                      max_chars=cfg.refiner_max_chars, compact=env.compact_trajectory)
print(f"{len(prompt):,} characters\n")
print(prompt[:900], "\n\n   [...]\n")
print(prompt[-700:])

5 nodes, 0 edges 

4,785 characters

## Environment
HotpotQA (distractor setting): answer a multi-hop question whose evidence is spread over two of ten context paragraphs. Tools search the paragraphs by word overlap, look up a paragraph by title, and submit a short answer span. Score is token F1 against the gold answer.

## Tools
- search: Search the context paragraphs. Returns the 3 paragraphs with the highest word overlap with the query.
- lookup: Return the full paragraph with the given title.
- finish: Submit the final answer (a short span such as an entity name, a date, or yes/no). Ends the episode.

## Current graph (JSON)
{
 "name": "hotpotqa_demo",
 "nodes": [
  {
   "id": "Start",
   "type": "STATE",
   "description": "Beginning of the task."
  },
  {
   "id": "End",
   "type": "STATE",
   "description": "Task complete."
  },
  {
   "id": "search",
   "type": "ACTION",
   "description": "Search the context paragrap 

   [...]

l and Jordi Valbuena, published in 2014 by Ludonau

To take the last step yourself (one call to the refiner model, a few cents), uncomment the cell below. It returns a
typed `EditSet`; `apply` builds the candidate graph without touching the original, and `diff` shows what changed.
`evolve` would now run the candidate on the validation split and keep it only if it scores at least as well.

In [17]:
# from pg.llm import make_llm
# from pg.refiner import propose_edits
# from analysis.graph_evolution import text_diff
#
# edits, usage = propose_edits(make_llm(cfg.refiner_model, cfg.temperature), prompt)
# candidate, warnings = skeleton.apply(edits)
# print(edits.rationale, "\n")
# print(text_diff(skeleton.diff(candidate)))